# Mapa de Geoanalitica Territorial - ITT (Ciudad Paraiso / Pulmon de Oriente)

Notebook de geoanalitica territorial. La unidad espacial de analisis es la **manzana catastral** (`data/catastral/MANZANAS_U_2024.shp`). El pipeline:

1. Carga capas catastrales reales (manzanas, terrenos, construcciones) desde `data/catastral/`.
2. Calcula indicadores por manzana e integra los eventos ITT (seguridad / ambiente).
3. Clasifica cada manzana en un nivel ITT (Activacion / Consolidacion / Transformacion / Escala).
4. Exporta artefactos intermedios a `data_processed_geo/` (`.geojson`, `.parquet`).
5. Genera un mapa `.html` interactivo y autonomo (Folium) con popups enriquecidos.

**Convenciones del proyecto:** entorno unico `uv`, codigo sin emojis, paleta Okabe-Ito accesible para daltonismo, CRS metrico EPSG:9377 y capa web EPSG:4326.

> IMPORTANTE - datos de eventos: hurtos, homicidios, rinas, lesionados, siniestralidad, SPA, VIF y censo arboreo NO estan aun en el proyecto. La celda 4 genera datos **sinteticos marcados como placeholder** para que el pipeline sea ejecutable de punta a punta. Al llegar los datos reales, se reemplaza unicamente esa celda.

## 1. Entorno y paleta Okabe-Ito

In [ ]:
# Dependencias: ejecutar en el entorno uv del proyecto.
#   uv run --with geopandas --with folium --with pyarrow jupyter lab
# o, para una celda suelta: uv run --with geopandas --with folium --with pyarrow python ...
import os
import math
import json
import numpy as np
import pandas as pd
import geopandas as gpd

# Paleta Okabe-Ito (accesible para daltonismo) - misma paleta que el resto del proyecto
OKI_AZUL     = "#0072B2"
OKI_NARANJA  = "#E69F00"
OKI_VERDE    = "#009E73"
OKI_AMARILLO = "#F0E442"
OKI_AZUL_C   = "#56B4E9"
OKI_BERMELL  = "#D55E00"
OKI_MORADO   = "#CC79A7"
OKI_GRIS     = "#333333"
BG = "#F4F6F9"

# Niveles ITT (color saturado y su version pastel para bandas / coropleta suave)
NIVEL_COLORS = {
    "Activacion":     OKI_BERMELL,
    "Consolidacion":  OKI_NARANJA,
    "Transformacion": OKI_VERDE,
    "Escala":         OKI_AZUL,
}
NIVEL_PASTEL = {
    "Activacion":     "#FFCDD2",
    "Consolidacion":  "#FFE0B2",
    "Transformacion": "#C8E6C9",
    "Escala":         "#BBDEFB",
}
# Color tematico por indicador de evento
EVENTO_COLORS = {
    "hurtos":        OKI_AZUL,
    "siniestros":    OKI_NARANJA,
    "rinas":         OKI_VERDE,
    "spa":           OKI_AMARILLO,
    "lesionados":    OKI_AZUL_C,
    "homicidios":    OKI_BERMELL,
    "vif":           OKI_MORADO,
}

# CRS del proyecto
CRS_WEB    = 4326   # WGS84, capa web
CRS_METRIC = 9377   # MAGNA-SIRGAS Origen Nacional, calculo metrico

print("Entorno listo. Paleta Okabe-Ito cargada.")

## 2. Rutas y carga de capas catastrales (datos reales)

In [ ]:
# Resolucion robusta de rutas. Funciona en local (uv) y en Google Colab.
# Se puede forzar con la variable de entorno CATASTRAL_DIR.
REQUERIDOS = ["MANZANAS_U_2024.shp", "TERRENOS_U_2024.shp", "CONSTRUCCION_U_2024.shp"]

def tiene_shapefiles(p):
    return bool(p) and all(os.path.isfile(os.path.join(p, f)) for f in REQUERIDOS)

def resolver_catastral():
    # 1) Variable de entorno explicita
    env = os.environ.get("CATASTRAL_DIR")
    candidatos = [env] if env else []
    # 2) Rutas locales (notebook en notebooks/, data un nivel arriba)
    candidatos += [
        os.path.join("..", "data", "catastral"),
        os.path.join("data", "catastral"),
        "catastral",
    ]
    # 3) Google Colab con Drive montado (ajusta la ruta a tu carpeta en Drive)
    candidatos += [
        "/content/data/catastral",
        "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/catastral",
    ]
    for p in candidatos:
        if tiene_shapefiles(p):
            return os.path.abspath(p)
    return None

CATASTRAL_DIR = resolver_catastral()

if CATASTRAL_DIR is None:
    # En Colab: ofrecer subir los shapefiles (los 6+ archivos por capa: shp, dbf, shx, prj, cpg...).
    try:
        from google.colab import files
        destino = "/content/data/catastral"
        os.makedirs(destino, exist_ok=True)
        print("No se encontro data/catastral. Sube los archivos de las 3 capas")
        print("(cada capa: .shp .dbf .shx .prj .cpg ...). Selecciona todos a la vez:")
        subidos = files.upload()
        for nombre, contenido in subidos.items():
            with open(os.path.join(destino, nombre), "wb") as fh:
                fh.write(contenido)
        CATASTRAL_DIR = destino if tiene_shapefiles(destino) else None
    except ImportError:
        pass

if CATASTRAL_DIR is None:
    raise FileNotFoundError(
        "No se encontraron los shapefiles catastrales (" + ", ".join(REQUERIDOS) + ").\n"
        "Local: ejecuta desde la carpeta notebooks/ del proyecto (data/catastral/ debe existir un nivel arriba).\n"
        "Colab: monta Google Drive y define CATASTRAL_DIR, o sube los archivos cuando se solicite.\n"
        "Tambien puedes fijar la ruta: os.environ['CATASTRAL_DIR'] = '/ruta/a/data/catastral'"
    )

# Carpeta de salida junto a la raiz del proyecto (o local al notebook en Colab)
OUT_DIR = os.path.abspath(os.path.join(os.path.dirname(CATASTRAL_DIR.rstrip(os.sep)), "..", "data_processed_geo"))
os.makedirs(OUT_DIR, exist_ok=True)

mz = gpd.read_file(os.path.join(CATASTRAL_DIR, "MANZANAS_U_2024.shp"))
terrenos = gpd.read_file(os.path.join(CATASTRAL_DIR, "TERRENOS_U_2024.shp"))
construcciones = gpd.read_file(os.path.join(CATASTRAL_DIR, "CONSTRUCCION_U_2024.shp"))

# Normalizar clave de union. CONSTRUCCION no trae IDMANZANA: se compone
# como COMUNA(2) + BARRIO(2) + MANZANA(4), igual que en MANZANAS y TERRENOS.
def asegurar_idmanzana(g):
    if "IDMANZANA" not in g.columns:
        g["IDMANZANA"] = (g["COMUNA"].astype(str) + g["BARRIO"].astype(str) + g["MANZANA"].astype(str))
    g["IDMANZANA"] = g["IDMANZANA"].astype(str).str.strip()
    return g

mz = asegurar_idmanzana(mz)
terrenos = asegurar_idmanzana(terrenos)
construcciones = asegurar_idmanzana(construcciones)

print("Catastral:", CATASTRAL_DIR)
print("Salida procesada:", OUT_DIR)
print(f"Manzanas: {len(mz)} | CRS: {mz.crs}")
print(f"Terrenos (predios): {len(terrenos)}")
print(f"Construcciones: {len(construcciones)}")

# Poligono de la zona de intervencion (Ciudad Paraiso). Se resuelve igual que la
# data catastral (local + Colab) y se reproyecta a WGS84 para los mapas.
def resolver_poligono():
    env = os.environ.get("POLIGONO_ZONA")
    cand = [env] if env else []
    cand += [
        os.path.join(os.path.dirname(CATASTRAL_DIR.rstrip(os.sep)), "geojson", "poligono_ciudad_Paraiso.geojson"),
        os.path.join("..", "data", "geojson", "poligono_ciudad_Paraiso.geojson"),
        os.path.join("data", "geojson", "poligono_ciudad_Paraiso.geojson"),
        "/content/data/geojson/poligono_ciudad_Paraiso.geojson",
        "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/geojson/poligono_ciudad_Paraiso.geojson",
    ]
    for p in cand:
        if p and os.path.isfile(p):
            return os.path.abspath(p)
    return None

_poly_path = resolver_poligono()
POLIGONO_ZONA = None
if _poly_path:
    POLIGONO_ZONA = gpd.read_file(_poly_path).to_crs(CRS_WEB)
    print("Poligono de zona:", _poly_path, "| features:", len(POLIGONO_ZONA))
else:
    print("Aviso: no se encontro poligono_ciudad_Paraiso.geojson; los mapas se dibujan sin el contorno de zona.")

# Capa de terrenos (predios) para usar como SOMBRA de fondo (contexto catastral).
def resolver_terrenos():
    env = os.environ.get("TERRENOS_GEOJSON")
    cand = [env] if env else []
    cand += [
        os.path.join(os.path.dirname(CATASTRAL_DIR.rstrip(os.sep)), "geojson", "TERRENOS_U_2024.geojson"),
        os.path.join("..", "data", "geojson", "TERRENOS_U_2024.geojson"),
        os.path.join("data", "geojson", "TERRENOS_U_2024.geojson"),
        "/content/data/geojson/TERRENOS_U_2024.geojson",
        "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/geojson/TERRENOS_U_2024.geojson",
    ]
    for p in cand:
        if p and os.path.isfile(p):
            return os.path.abspath(p)
    return None

_terr_path = resolver_terrenos()
TERRENOS_ZONA = None
if _terr_path:
    TERRENOS_ZONA = gpd.read_file(_terr_path).to_crs(CRS_WEB)
    print("Terrenos (sombra):", _terr_path, "| predios:", len(TERRENOS_ZONA))
else:
    print("Aviso: no se encontro TERRENOS_U_2024.geojson; los mapas se dibujan sin la sombra de predios.")

## 3. Indicadores reales por manzana

In [ ]:
# Area de manzana en metros (reproyectada a CRS metrico)
mz_m = mz.to_crs(CRS_METRIC)
mz = mz.copy()
mz["area_m2"] = mz_m.geometry.area.values

# Predios por manzana
pred_por_mz = terrenos.groupby("IDMANZANA").size().rename("num_predios")

# Construcciones: numero y pisos promedio por manzana
construcciones["NPISOS"] = pd.to_numeric(construcciones.get("NPISOS"), errors="coerce")
const_agg = construcciones.groupby("IDMANZANA").agg(
    num_construcciones=("NPISOS", "size"),
    pisos_prom=("NPISOS", "mean"),
    pisos_max=("NPISOS", "max"),
)

mz = mz.merge(pred_por_mz, on="IDMANZANA", how="left")
mz = mz.merge(const_agg, on="IDMANZANA", how="left")
mz[["num_predios", "num_construcciones"]] = mz[["num_predios", "num_construcciones"]].fillna(0).astype(int)
mz["pisos_prom"] = mz["pisos_prom"].fillna(0).round(2)
mz["pisos_max"] = mz["pisos_max"].fillna(0).astype(int)

# Densidad predial (predios por hectarea)
mz["densidad_predial"] = (mz["num_predios"] / (mz["area_m2"] / 10000.0)).round(1)

mz[["IDMANZANA", "COMUNA", "BARRIO", "area_m2", "num_predios", "num_construcciones", "pisos_prom", "densidad_predial"]].head(10)

## 4. Eventos ITT (PLACEHOLDER - reemplazar con datos reales)

Esta celda es el UNICO punto donde se inyectan datos sinteticos. Cuando lleguen los eventos reales (hurtos, homicidios, rinas, lesionados, siniestralidad, SPA, VIF) y el censo arboreo, reemplazar el contenido por la carga/cruce espacial real a nivel de manzana (join por `IDMANZANA` o join espacial por coordenadas), manteniendo los mismos nombres de columna.

In [ ]:
# ADVERTENCIA: datos sinteticos de demostracion. NO usar para decisiones.
DATOS_SINTETICOS = True
rng = np.random.default_rng(42)
n = len(mz)

# Conteos de eventos por manzana (periodo actual T0 y anterior T-1 para tendencia)
for col in ["hurtos", "siniestros", "rinas", "spa", "lesionados", "homicidios", "vif"]:
    base = rng.integers(0, 12, size=n)
    mz[col + "_t0"] = base
    mz[col + "_t1"] = np.clip(base + rng.integers(-4, 5, size=n), 0, None)

# Censo arboreo (arboles por manzana) y NDVI promedio [0..1]
mz["arboles"] = rng.integers(0, 60, size=n)
mz["ndvi"] = (rng.uniform(0.1, 0.6, size=n)).round(3)

print("Eventos sinteticos generados (placeholder). Columnas de evento:")
print([c for c in mz.columns if c.endswith("_t0") or c in ("arboles", "ndvi")])

## 5. Indice ITT y clasificacion en 4 niveles

In [ ]:
def norm_inv(s):
    """Normaliza 0..1 e invierte: mas eventos negativos -> menor score."""
    s = s.astype(float)
    rng_ = s.max() - s.min()
    if rng_ == 0:
        return pd.Series(1.0, index=s.index)
    return 1.0 - (s - s.min()) / rng_

def norm_dir(s):
    """Normaliza 0..1 directo: mas es mejor (ej. arbolado)."""
    s = s.astype(float)
    rng_ = s.max() - s.min()
    if rng_ == 0:
        return pd.Series(1.0, index=s.index)
    return (s - s.min()) / rng_

# Dimensiones ITT (ejemplo 5 dim): seguridad, convivencia, movilidad, ambiente, consolidacion urbana
seguridad   = norm_inv(mz["hurtos_t0"] + mz["homicidios_t0"])
convivencia = norm_inv(mz["rinas_t0"] + mz["vif_t0"] + mz["spa_t0"])
movilidad   = norm_inv(mz["siniestros_t0"] + mz["lesionados_t0"])
ambiente    = (norm_dir(mz["arboles"]) + norm_dir(mz["ndvi"])) / 2.0
urbano      = norm_dir(mz["pisos_prom"]) 

mz["itt"] = ((seguridad + convivencia + movilidad + ambiente + urbano) / 5.0 * 100).round(1)

def nivel(itt):
    if itt < 40:  return "Activacion"
    if itt < 60:  return "Consolidacion"
    if itt < 80:  return "Transformacion"
    return "Escala"

mz["nivel"] = mz["itt"].map(nivel)
mz["color_nivel"] = mz["nivel"].map(NIVEL_COLORS)

print(mz["nivel"].value_counts().to_string())
print("\nITT min/med/max:", mz["itt"].min(), mz["itt"].median(), mz["itt"].max())

## 6. Exportar artefactos procesados (data_processed_geo/)

In [ ]:
mz_web = mz.to_crs(CRS_WEB)

geojson_path = os.path.join(OUT_DIR, "manzanas_itt.geojson")
parquet_path = os.path.join(OUT_DIR, "manzanas_itt.parquet")

mz_web.to_file(geojson_path, driver="GeoJSON")
try:
    mz_web.to_parquet(parquet_path)
    print("Parquet:", parquet_path)
except Exception as e:
    print("Parquet no generado (requiere pyarrow):", e)

print("GeoJSON:", geojson_path)
print("Manzanas exportadas:", len(mz_web))

## 7. Popups enriquecidos (HTML/CSS + mini-graficas SVG)

In [ ]:
def svg_barras(valores, colores, w=180, h=70):
    """Mini grafico de barras en SVG inline (sin dependencias JS en el popup)."""
    vmax = max(valores) if max(valores) > 0 else 1
    bw = w / (len(valores) * 1.5)
    gap = bw * 0.5
    barras = []
    for i, (v, c) in enumerate(zip(valores, colores)):
        bh = (v / vmax) * (h - 18)
        x = gap + i * (bw + gap)
        y = h - bh - 12
        barras.append(
            f'<rect x="{x:.1f}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" '
            f'fill="{c}" stroke="{OKI_GRIS}" stroke-width="0.5" rx="1"/>'
            f'<text x="{x + bw/2:.1f}" y="{h-2}" font-size="8" fill="{OKI_GRIS}" '
            f'text-anchor="middle">{v}</text>'
        )
    return f'<svg width="{w}" height="{h}" style="background:{BG};border-radius:4px">' + "".join(barras) + "</svg>"

def pct_var(t0, t1):
    if t1 == 0:
        return "s/d" if t0 == 0 else "+100%"
    return f"{(t0 - t1) / t1 * 100:+.0f}%"

def popup_html(r):
    eventos = ["hurtos", "siniestros", "rinas", "spa", "lesionados", "homicidios", "vif"]
    labels = ["Hurtos", "Siniestros", "Rinas", "SPA", "Lesion.", "Homic.", "VIF"]
    vals = [int(r[e + "_t0"]) for e in eventos]
    cols = [EVENTO_COLORS[e] for e in eventos]
    svg = svg_barras(vals, cols)
    filas = "".join(
        f'<tr><td style="padding:1px 6px">{lab}</td>'
        f'<td style="text-align:right;padding:1px 6px"><b>{r[e+"_t0"]}</b></td>'
        f'<td style="text-align:right;padding:1px 6px;color:{OKI_GRIS}">{pct_var(r[e+"_t0"], r[e+"_t1"])}</td></tr>'
        for e, lab in zip(eventos, labels)
    )
    nivel_c = NIVEL_COLORS[r["nivel"]]
    aviso = ('<div style="color:#b00;font-size:9px;margin-top:4px">Datos de evento SINTETICOS (demo)</div>'
             if DATOS_SINTETICOS else "")
    return (
        f'<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
        f'background:{BG};padding:8px;border-radius:6px;min-width:230px">'
        f'<div style="font-size:13px;font-weight:700;color:{OKI_GRIS}">Manzana {r["MANZANA"]} '
        f'<span style="font-weight:400;color:#777">({r["IDMANZANA"]})</span></div>'
        f'<div style="color:#555">Comuna {r["COMUNA"]} - Barrio {r["BARRIO"]}</div>'
        f'<div style="margin:4px 0"><span style="background:{nivel_c};color:#fff;'
        f'padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700">'
        f'{r["nivel"]} - ITT {r["itt"]}</span></div>'
        f'<div style="color:#555;font-size:11px">Area: {r["area_m2"]/10000:.2f} ha - '
        f'Predios: {r["num_predios"]} - Pisos prom: {r["pisos_prom"]}</div>'
        f'<div style="margin-top:6px;font-size:10px;color:{OKI_GRIS};font-weight:700">'
        f'Eventos T0 por tipo</div>{svg}'
        f'<table style="font-size:11px;border-collapse:collapse;margin-top:4px;width:100%">'
        f'<tr style="color:{OKI_GRIS};border-bottom:1px solid #ccc">'
        f'<td style="padding:1px 6px">Evento</td><td style="text-align:right;padding:1px 6px">T0</td>'
        f'<td style="text-align:right;padding:1px 6px">vs T-1</td></tr>{filas}</table>{aviso}</div>'
    )

print("Funciones de popup listas. Ejemplo de longitud HTML:", len(popup_html(mz.iloc[0])), "chars")

## 8. Construccion del mapa interactivo (Folium)

## Exportacion de lamina (deck)

Funcion reutilizable `agregar_export(mapa, titulo, slug)` que inyecta en cada mapa UN boton (Exportar lamina (deck)). Al pulsar, captura el contenedor del mapa a una imagen con html2canvas (ocultando tiles base por CORS, botones, selector de capas, panel flotante y slider de calor) y la compone dentro de una lamina HTML con estructura de deck del PMU: encabezado dorado con titulo, cuerpo en 2 columnas (izquierda con el resumen/cifras clonado del panel existente, derecha con el mapa) y pie con corte y fuente. Luego captura la lamina y la exporta a PDF de una pagina (landscape) con dialogo Guardar como (showSaveFilePicker) y fallback a descarga. Nombre deck_<slug>_<AAAA-MM-DD>.pdf. Nota CORS: los tiles base de Google/Esri no se rasterizan (se ocultan y se usa fondo claro); la imagen del mapa muestra contorno, capas activas y overlays.

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      // 2) fitBounds al poligono para centrar/proporcionar la lamina.
      try { if(window[MAP_ID]){ window[MAP_ID].fitBounds(ZONA_BOUNDS, {padding:[10,10]}); } } catch(e){}
      // 3) Esperar a que el mapa reencuadre y las capas se redibujen.
      setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
      }, 800);
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa


In [ ]:
import folium
from folium.plugins import HeatMap

centro = [mz_web.geometry.union_all().centroid.y, mz_web.geometry.union_all().centroid.x]
m = folium.Map(location=centro, zoom_start=16, tiles=None, control_scale=True,
                   zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)

# 4 capas base conmutables (OSM activa por defecto)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)

# Capa 1: contorno territorial (activa al inicio)
contorno = folium.FeatureGroup(name="Contorno territorial", show=True)
folium.GeoJson(
    mz_web.to_json(),
    style_function=lambda f: {"color": OKI_GRIS, "weight": 1.2, "fill": False},
).add_to(contorno)
contorno.add_to(m)

# Capa 2: coropleta por nivel ITT (apagada al inicio)
coropleta = folium.FeatureGroup(name="Coropleta ITT (nivel)", show=False)
for _, r in mz_web.iterrows():
    folium.GeoJson(
        r["geometry"].__geo_interface__,
        style_function=lambda f, c=NIVEL_PASTEL[r["nivel"]], b=NIVEL_COLORS[r["nivel"]]: {
            "fillColor": c, "color": b, "weight": 1.5, "fillOpacity": 0.7},
        popup=folium.Popup(popup_html(r), max_width=280),
        tooltip=f'Manzana {r["MANZANA"]} - {r["nivel"]}',
    ).add_to(coropleta)
coropleta.add_to(m)

# Capa del poligono de la zona de intervencion (Ciudad Paraiso), borde dorado.
if POLIGONO_ZONA is not None:
    zona_fg = folium.FeatureGroup(name="Poligono Ciudad Paraiso", show=True)
    folium.GeoJson(
        POLIGONO_ZONA.to_json(),
        style_function=lambda f: {"color": "#8A6D1A", "weight": 5,
                                  "dashArray": "10 6", "fill": False},
        tooltip="Zona de intervencion: Ciudad Paraiso",
    ).add_to(zona_fg)
    zona_fg.add_to(m)
# Capa de SOMBRA: predios en gris translucido como contexto catastral.
# Sin popup; apagada al inicio. Da textura de la trama de predios bajo las coropletas.
if TERRENOS_ZONA is not None:
    sombra_fg = folium.FeatureGroup(name="Predios (sombra)", show=False)
    folium.GeoJson(
        TERRENOS_ZONA.to_json(),
        style_function=lambda f: {"fillColor": "#333333", "color": "#555555",
                                  "weight": 0.3, "fillOpacity": 0.18},
    ).add_to(sombra_fg)
    sombra_fg.add_to(m)
print("Mapa base + contorno + coropleta construidos.")

In [ ]:
# Capa 3: coropleta de densidad de hurtos por manzana (gradiente azul por cantidad).
# Representa mejor el territorio que un punto en el centroide. Con datos reales,
# usar el conteo real de hurtos por manzana (o un join espacial de los eventos).
def _rgb_hex(h):
    h = h.lstrip("#")
    return tuple(int(h[k:k + 2], 16) for k in (0, 2, 4))
def _grad_azul(pct):
    r, g, b = _rgb_hex(EVENTO_COLORS["hurtos"])
    t = max(0.0, min(1.0, pct))
    return "#%02X%02X%02X" % (int(255 + (r - 255) * t), int(255 + (g - 255) * t), int(255 + (b - 255) * t))
# --- diagrama de torta SVG: composicion de delitos de la manzana ---
DELITOS = ["hurtos", "homicidios", "rinas", "lesionados", "siniestros", "spa"]
DELITO_LABEL = {"hurtos": "Hurtos", "homicidios": "Homicidios", "rinas": "Rinas",
                "lesionados": "Lesionados", "siniestros": "Siniestros", "spa": "SPA"}
def svg_torta(valores, colores, size=120):
    total = float(sum(valores))
    cx = cy = size / 2.0
    rad = size / 2.0 - 4
    if total <= 0:
        return ('<svg width="%d" height="%d"><circle cx="%.1f" cy="%.1f" r="%.1f" '
                'fill="#E8E8E8" stroke="%s" stroke-width="0.5"/>'
                '<text x="%.1f" y="%.1f" font-size="9" fill="%s" text-anchor="middle">sin datos</text>'
                '</svg>') % (size, size, cx, cy, rad, OKI_GRIS, cx, cy, OKI_GRIS)
    partes = []
    ang0 = -90.0
    for v, c in zip(valores, colores):
        if v <= 0:
            continue
        frac = v / total
        ang1 = ang0 + frac * 360.0
        x0 = cx + rad * math.cos(math.radians(ang0))
        y0 = cy + rad * math.sin(math.radians(ang0))
        x1 = cx + rad * math.cos(math.radians(ang1))
        y1 = cy + rad * math.sin(math.radians(ang1))
        large = 1 if (ang1 - ang0) > 180 else 0
        if frac >= 0.999:
            partes.append('<circle cx="%.1f" cy="%.1f" r="%.1f" fill="%s" stroke="#fff" stroke-width="0.8"/>'
                          % (cx, cy, rad, c))
        else:
            partes.append('<path d="M%.1f,%.1f L%.1f,%.1f A%.1f,%.1f 0 %d,1 %.1f,%.1f Z" '
                          'fill="%s" stroke="#fff" stroke-width="0.8"/>'
                          % (cx, cy, x0, y0, rad, rad, large, x1, y1, c))
        ang0 = ang1
    return '<svg width="%d" height="%d" style="background:%s;border-radius:6px">%s</svg>' % (
        size, size, BG, "".join(partes))
def popup_hurtos_html(r):
    vals = [int(r[d + "_t0"]) for d in DELITOS]
    cols = [EVENTO_COLORS[d] for d in DELITOS]
    torta = svg_torta(vals, cols)
    total = sum(vals)
    leyenda = "".join(
        '<div style="display:flex;align-items:center;gap:5px;margin:1px 0">'
        '<i style="width:10px;height:10px;background:%s;display:inline-block;border-radius:2px;'
        'border:0.5px solid %s"></i><span style="font-size:10px">%s</span></div>'
        % (EVENTO_COLORS[d], OKI_GRIS, DELITO_LABEL[d]) for d in DELITOS
    )
    filas = "".join(
        '<tr><td style="padding:1px 6px">%s</td>'
        '<td style="text-align:right;padding:1px 6px"><b>%d</b></td>'
        '<td style="text-align:right;padding:1px 6px">%s</td>'
        '<td style="text-align:right;padding:1px 6px;color:%s">%s</td></tr>'
        % (DELITO_LABEL[d], int(r[d + "_t0"]),
           ("%.0f%%" % (100.0 * int(r[d + "_t0"]) / total)) if total else "-",
           OKI_GRIS, pct_var(int(r[d + "_t0"]), int(r[d + "_t1"])))
        for d in DELITOS
    )
    return (
        '<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
        'background:%s;padding:8px;border-radius:6px;min-width:260px">'
        '<div style="font-size:13px;font-weight:700;color:%s">Delitos - Manzana %s '
        '<span style="font-weight:400;color:#777">(%s)</span></div>'
        '<div style="color:#555">Comuna %s - Barrio %s - Total T0: <b>%d</b></div>'
        '<div style="display:flex;gap:10px;align-items:center;margin-top:6px">'
        '%s<div>%s</div></div>'
        '<table style="font-size:11px;border-collapse:collapse;margin-top:6px;width:100%%">'
        '<tr style="color:%s;border-bottom:1px solid #ccc">'
        '<td style="padding:1px 6px">Delito</td>'
        '<td style="text-align:right;padding:1px 6px">T0</td>'
        '<td style="text-align:right;padding:1px 6px">%% del total</td>'
        '<td style="text-align:right;padding:1px 6px">vs T-1</td></tr>%s</table>'
        '<div style="font-size:9px;color:#b00;margin-top:4px">Datos de evento SINTETICOS (demo)</div>'
        '</div>'
    ) % (BG, OKI_GRIS, r["MANZANA"], r["IDMANZANA"] if "IDMANZANA" in r else "",
         r["COMUNA"], r["BARRIO"], total, torta, leyenda, OKI_GRIS, filas)
_hmax = float(mz_web["hurtos_t0"].max()) or 1.0
hurtos_cp = folium.FeatureGroup(name="Hurtos (densidad por manzana)", show=False)
for _, r in mz_web.iterrows():
    val = int(r["hurtos_t0"])
    fill = _grad_azul(val / _hmax) if val > 0 else "#E8E8E8"
    folium.GeoJson(
        r["geometry"].__geo_interface__,
        style_function=(lambda f, c=fill: {"fillColor": c, "color": OKI_GRIS,
                                            "weight": 1.0, "fillOpacity": 0.8}),
        popup=folium.Popup(popup_hurtos_html(r), max_width=320),
        tooltip=f'Hurtos T0: {val} (manzana {r["MANZANA"]})',
    ).add_to(hurtos_cp)
hurtos_cp.add_to(m)

# Capa 4: VIF como HeatMap (privacidad: NUNCA punto individual)
vif = folium.FeatureGroup(name="VIF (mapa de calor)", show=False)
pts = []
for _, r in mz_web.iterrows():
    cen = r["geometry"].centroid
    peso = float(r["vif_t0"])
    if peso > 0:
        pts.append([cen.y, cen.x, peso])
if pts:
    HeatMap(pts, radius=28, blur=22, min_opacity=0.3).add_to(vif)
vif.add_to(m)

print("Capas de densidad de hurtos y HeatMap VIF agregadas. Puntos VIF:", len(pts))

## 9. Leyenda, control de capas y exportacion del HTML

In [ ]:
# Leyenda de niveles ITT (HTML flotante)
items = "".join(
    f'<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    f'<i style="width:14px;height:14px;background:{c};border:1px solid {OKI_GRIS};'
    f'display:inline-block;border-radius:3px"></i><span>{k}</span></div>'
    for k, c in NIVEL_COLORS.items()
)
aviso = ('<div style="color:#b00;font-size:10px;margin-top:4px">Eventos: datos sinteticos (demo)</div>'
         if DATOS_SINTETICOS else "")
legend = (
    f'<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:{BG};'
    f'padding:10px 12px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
    f'font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:{OKI_GRIS}">'
    f'<div style="font-weight:700;margin-bottom:4px">Nivel ITT</div>{items}{aviso}</div>'
)
m.get_root().html.add_child(folium.Element(legend))

if POLIGONO_ZONA is not None:
    m.fit_bounds([[3.442445, -76.53259], [3.44791, -76.526004]], padding=(30, 30))
folium.LayerControl(collapsed=False, position="topright").add_to(m)

agregar_export(m, "Mapa ITT", "itt")
html_path = os.path.join(OUT_DIR, "mapa_geoanalitica_itt.html")
m.save(html_path)
print("Mapa exportado:", html_path)
m

## 10. Resumen analitico de hallazgos territoriales

In [ ]:
resumen = {
    "manzanas_analizadas": int(len(mz)),
    "predios_totales": int(mz["num_predios"].sum()),
    "construcciones_totales": int(mz["num_construcciones"].sum()),
    "itt_promedio": float(mz["itt"].mean().round(1)),
    "por_nivel": mz["nivel"].value_counts().to_dict(),
    "datos_eventos_sinteticos": DATOS_SINTETICOS,
}
print(json.dumps(resumen, ensure_ascii=False, indent=2))

print("\nHallazgos (sobre indicadores catastrales reales):")
top_den = mz.sort_values("densidad_predial", ascending=False).head(3)
for _, r in top_den.iterrows():
    print(f"  - Manzana {r['MANZANA']} ({r['IDMANZANA']}): {r['densidad_predial']} predios/ha, "
          f"{r['num_predios']} predios, pisos prom {r['pisos_prom']}")
print("\nNota: la clasificacion ITT depende de los eventos; con datos sinteticos los niveles")
print("son ilustrativos. Los indicadores catastrales (predios, area, pisos, densidad) SI son reales.")

## 11. Avance de caracterizacion por manzana (datos reales)
Nueva tematica sobre datos reales de la encuesta (`data/*.xlsx`). Se reutiliza la logica de geolocalizacion (GPS por prioridad Ficha > Hogares > Salud > Empresas, y direccion declarada sobre grilla calle/carrera) y de clasificacion de avance por eje del notebook de visitas. El avance se separa en 4 ejes tematicos: **Ficha, Hogares, Salud y Empresas**. En el mapa solo se publican agregados por manzana (nunca puntos de personas).

### 11a. Carga de los Excel de la encuesta

In [ ]:
# SECCION_11_MARK
# 11a. Carga de los 5 Excel de la encuesta (datos reales en data/).
# Lectura directa de los archivos locales (sin modelo S3). El archivo de
# caracterizacion lleva acento en el nombre: se localiza por glob con 'caracteriz'.
import glob, io, math, re, unicodedata

def _resolver_data_dir():
    cands = [os.path.join("..", "data"), os.path.join("data"), "."]
    env = os.environ.get("DATA_DIR")
    if env:
        cands = [env] + cands
    for c in cands:
        if c and os.path.isfile(os.path.join(c, "ListadoCiudadanosCompleto.xlsx")):
            return os.path.abspath(c)
    raise FileNotFoundError("No se encontro data/ con los Excel de la encuesta.")

DATA_DIR = _resolver_data_dir()

def _primer_match(token):
    fs = glob.glob(os.path.join(DATA_DIR, "*" + token + "*"))
    if not fs:
        raise FileNotFoundError("No se encontro Excel con token '" + token + "'")
    return fs[0]

# Lectura como texto (dtype=str) para preservar documentos y coordenadas tal cual.
mae = pd.read_excel(_primer_match("ListadoCiudadanosCompleto"), sheet_name="Ciudadanos", dtype=str)
hog = pd.read_excel(_primer_match("ListadoHogaresIntegrantes"), sheet_name="Hogares", dtype=str)
fic = pd.read_excel(_primer_match("caracteriz"), sheet_name=0, dtype=str)
prog = pd.read_excel(_primer_match("respuestas_programas"), sheet_name=None, dtype=str)

# Nombres de columnas clave (con acentos tal como vienen en los Excel).
COL_DOC_MAE = "Numero de documento" if "Numero de documento" in mae.columns else "N\u00famero de documento"
COL_DOC_FIC = "Numero de identificacion" if "Numero de identificacion" in fic.columns else "N\u00famero de identificaci\u00f3n"

print("DATA_DIR:", DATA_DIR)
print("Ciudadanos (padron):", len(mae), "| docs unicos:", mae[COL_DOC_MAE].nunique())
print("Hogares integrantes:", len(hog))
print("Ficha caracterizacion:", len(fic))
for k, v in prog.items():
    print("  Programa", k.replace("Programa ", ""), "->", len(v), "filas")


### 11b. Geolocalizacion y asignacion a manzana

In [ ]:
# 11b. Geolocalizacion de cada ciudadano y asignacion a manzana catastral.
# Logica portada TAL CUAL del notebook de visitas (Mapa_Visitas_Colab_v1):
#   GPS por prioridad de fuente Ficha > Hogares > Salud > Empresas, y, si no hay
#   GPS, estimacion de la direccion declarada sobre una grilla calle/carrera
#   ajustada por minimos cuadrados con los puntos GPS de ficha. Luego punto-en-
#   poligono sobre las manzanas catastrales reproyectadas a WGS84 (EPSG:4326).

# --- utilidades de texto (portadas de etl_handler.py) ---
def _s(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    x = str(x).strip()
    return x or None

# --- parseo de coordenadas y direcciones (portado tal cual) ---
PAT = re.compile(
    r"(?P<t>avenida carrera|calle|cll|cl|carrera|cra|cr|kr|kra)\.?\s*"
    r"(?P<m>\d{1,3})\s*(?P<ma>[a-zA-Z])?\s*(?:#|no\.?|n\u00b0|n\u00famero|numero|nro\.?)?"
    r"\s*[-\s]*(?P<c>\d{1,3})\s*(?P<ca>[a-zA-Z])?\s*(?:#|-|\u2013|\s)\s*(?P<h>\d{1,3})\b",
    re.I,
)
CALI_BBOX = (3.30, 3.60, -76.65, -76.40)
ZONA_BBOX = (3.425, 3.465, -76.545, -76.515)

def parse_addr(txt):
    """-> (norte, este, tipo) en indices de calle/carrera, o None."""
    if not isinstance(txt, str):
        return None
    mt = PAT.search(txt.split(",")[0])
    if not mt:
        return None
    t = "calle" if mt["t"].lower() in ("calle", "cll", "cl") else "carrera"
    main = int(mt["m"]) + (0.5 if mt["ma"] else 0)
    cross = int(mt["c"]) + (0.5 if mt["ca"] else 0)
    h = int(mt["h"])
    if h > 200:
        h //= 10
    return (cross + min(h, 99) / 100, main, t) if t == "carrera" else (main, cross + min(h, 99) / 100, t)

def parse_coord(v):
    v = _s(v)
    if not v:
        return None
    mt = re.findall(r"-?\d+\.\d+", v)
    if len(mt) < 2:
        return None
    la, lo = float(mt[0]), float(mt[1])
    return (la, lo) if (-90 <= la <= 90 and -180 <= lo <= 180) else None

def _hav(a, b):
    return math.hypot((a[0] - b[0]) * 111000,
                      (a[1] - b[1]) * 111000 * math.cos(math.radians(3.44)))

# --- construccion de ubicacion por persona (GPS por prioridad + direcciones) ---
EJES = ["ficha", "hogares", "salud", "empresas"]
FUENTES_GPS = [
    ("ficha", fic, COL_DOC_FIC),
    ("hogares", prog["Programa Hogares"], COL_DOC_FIC),
    ("salud", prog["Programa Salud"], COL_DOC_FIC),
    ("empresas", prog["Programa Empresas"], COL_DOC_FIC),
]

def build_ubicacion():
    gps, ubic_txt = {}, {}
    for nombre, df, kcol in FUENTES_GPS:
        if "Coordenadas" not in df.columns:
            continue
        d2 = df.drop_duplicates(kcol, keep="last")
        ucol = "Ubicacion" if "Ubicacion" in d2.columns else ("Ubicaci\u00f3n" if "Ubicaci\u00f3n" in d2.columns else None)
        for i, row in d2.iterrows():
            doc = row[kcol]
            cc = parse_coord(row.get("Coordenadas"))
            if cc:
                gps.setdefault(doc, []).append((nombre, cc))
            if ucol and _s(row.get(ucol)):
                ubic_txt.setdefault(doc, _s(row.get(ucol)))
    mae_i = mae.drop_duplicates(COL_DOC_MAE).set_index(COL_DOC_MAE)
    col_dir = "Direccion" if "Direccion" in mae.columns else "Direcci\u00f3n"
    col_comp = "Complemento de direccion" if "Complemento de direccion" in mae.columns else "Complemento de direcci\u00f3n"
    rows = []
    docs = list(mae_i.index)
    for doc in docs:
        lst = gps.get(doc, [])
        en_cali = [(n, c) for n, c in lst
                   if CALI_BBOX[0] <= c[0] <= CALI_BBOX[1] and CALI_BBOX[2] <= c[1] <= CALI_BBOX[3]]
        lat = lon = None
        fuente = None
        if en_cali:
            fuente, (lat, lon) = en_cali[0]
        pad = None
        if doc in mae_i.index:
            pad = " ".join(x for x in [_s(mae_i.at[doc, col_dir]) if col_dir in mae_i.columns else None,
                                       _s(mae_i.at[doc, col_comp]) if col_comp in mae_i.columns else None] if x) or None
        rows.append(dict(doc=doc, lat=lat, lon=lon, fuente=fuente,
                         dir_padron=pad, ubic_form=ubic_txt.get(doc),
                         declarada=bool(pad)))
    return pd.DataFrame(rows)

U = build_ubicacion()

def _addr_parse_row(r):
    for c in ("ubic_form", "dir_padron"):
        p = parse_addr(r[c])
        if p:
            return p
    return None

# --- ajuste de la grilla calle/carrera por minimos cuadrados (portado) ---
def fit_grid(U):
    U["p"] = U.apply(_addr_parse_row, axis=1)
    tr = U[U.p.notna() & U.lat.notna() & (U.fuente == "ficha") & U.declarada]
    if len(tr) < 30:
        return None, None, {"puntos": int(len(tr)), "aviso": "Muy pocos puntos para ajustar la grilla"}
    X = np.array([[1, p[0], p[1]] for p in tr.p])
    la, lo = tr.lat.values.astype(float), tr.lon.values.astype(float)
    mask = np.ones(len(tr), bool)
    dd = np.zeros(len(tr))
    for _ in range(5):
        cl = np.linalg.lstsq(X[mask], la[mask], rcond=None)[0]
        co = np.linalg.lstsq(X[mask], lo[mask], rcond=None)[0]
        dd = np.hypot((X @ cl - la) * 111000, (X @ co - lo) * 111000 * np.cos(np.radians(3.44)))
        mask = dd <= max(np.percentile(dd[mask], 90), 40)
    return cl, co, {"puntos_entrenamiento": int(len(tr)), "inliers": int(mask.sum()),
                    "error_mediano_m": round(float(np.median(dd[mask])), 1)}

def apply_grid(U, cl, co):
    U["ubic"] = np.where(U.lat.notna(), "GPS", "NINGUNA")
    if cl is None:
        return
    for i, r in U[U.lat.isna()].iterrows():
        p = r.p
        if p:
            la = float(cl @ np.array([1, p[0], p[1]]))
            lo = float(co @ np.array([1, p[0], p[1]]))
            if ZONA_BBOX[0] <= la <= ZONA_BBOX[1] and ZONA_BBOX[2] <= lo <= ZONA_BBOX[3]:
                U.loc[i, ["lat", "lon", "ubic"]] = [round(la, 6), round(lo, 6), "APROX"]

CL, CO, GRID_INFO = fit_grid(U)
apply_grid(U, CL, CO)

# --- asignacion a manzana por punto-en-poligono sobre manzanas en WGS84 ---
mz_wgs = mz.to_crs(CRS_WEB)[["IDMANZANA", "COMUNA", "BARRIO", "MANZANA", "geometry"]].copy()

def _pip(x, y, ring):
    c, n = False, len(ring)
    for i in range(n):
        x1, y1 = ring[i]
        x2, y2 = ring[(i + 1) % n]
        if (y1 > y) != (y2 > y) and x < (x2 - x1) * (y - y1) / (y2 - y1) + x1:
            c = not c
    return c

def _rings_de_geom(geom):
    gj = geom.__geo_interface__
    if gj["type"] == "Polygon":
        return [gj["coordinates"][0]]
    if gj["type"] == "MultiPolygon":
        return [poly[0] for poly in gj["coordinates"]]
    return []

MZ_RINGS = [(row["IDMANZANA"], _rings_de_geom(row["geometry"])) for _, row in mz_wgs.iterrows()]

# Distancia (m) de un punto al borde de un anillo (portada de etl_handler.py).
# Permite el "snap al borde": asignar a la manzana mas cercana los puntos que
# caen justo fuera del poligono (p. ej. sobre la calle) hasta SNAP_METROS.
SNAP_METROS = 25.0
def _dist_ring_m(x, y, ring):
    best = 1e18
    for i in range(len(ring)):
        x1, y1 = ring[i]
        x2, y2 = ring[(i + 1) % len(ring)]
        dx, dy = x2 - x1, y2 - y1
        t = max(0, min(1, ((x - x1) * dx + (y - y1) * dy) / (dx * dx + dy * dy or 1)))
        best = min(best, math.hypot((x - x1 - t * dx) * 111320 * 0.9984, (y - y1 - t * dy) * 110574))
    return best
def asignar_manzana(lat, lon):
    """Devuelve (IDMANZANA, calidad, dist_m). calidad: DENTRO / APROXIMADA / None."""
    if lat is None or (isinstance(lat, float) and np.isnan(lat)):
        return (None, None, None)
    for idm, rings in MZ_RINGS:
        if any(_pip(lon, lat, ring) for ring in rings):
            return (idm, "DENTRO", 0.0)
    mejor_id, mejor_d = None, 1e18
    for idm, rings in MZ_RINGS:
        d = min((_dist_ring_m(lon, lat, ring) for ring in rings), default=1e18)
        if d < mejor_d:
            mejor_id, mejor_d = idm, d
    if mejor_id is not None and mejor_d <= SNAP_METROS:
        return (mejor_id, "APROXIMADA", round(mejor_d, 1))
    return (None, None, round(mejor_d, 1) if mejor_id is not None else None)
_asig = [asignar_manzana(r.lat, r.lon) for r in U.itertuples()]
U["IDMANZANA"] = [a[0] for a in _asig]
U["mz_calidad"] = [a[1] for a in _asig]
U["mz_dist_m"] = [a[2] for a in _asig]
n_ubic = int(U["lat"].notna().sum())
n_gps = int((U["ubic"] == "GPS").sum())
n_aprox = int((U["ubic"] == "APROX").sum())
n_mz = int(U["IDMANZANA"].notna().sum())
n_dentro = int((U["mz_calidad"] == "DENTRO").sum())
n_borde = int((U["mz_calidad"] == "APROXIMADA").sum())
print("Grilla:", GRID_INFO)
print("Ciudadanos totales:", len(U))
print("Ubicados (lat/lon):", n_ubic, "| GPS:", n_gps, "| APROX (grilla):", n_aprox)
print("Con manzana asignada:", n_mz, "| dentro del poligono:", n_dentro,
      "| por snap al borde (<= %dm):" % SNAP_METROS, n_borde)


### 11c. Estado por eje y agregacion por manzana

In [ ]:
# 11c. Estado por eje (Ficha, Hogares, Salud, Empresas) y agregacion por manzana.
# Estado por eje de cada ciudadano (regla pstate del notebook de visitas):
#   '% de avance' == 100 -> COMPLETO ; == 0 -> VACIO ; resto -> PARCIAL.
# El eje Ficha es presencia en la caracterizacion: aparece -> COMPLETO, si no -> VACIO.

def pstate(df, kcol):
    """Serie doc -> estado (COMPLETO/PARCIAL/VACIO) segun '% de avance'."""
    d = df.copy()
    d["av"] = pd.to_numeric(d["% de avance"], errors="coerce")
    d["doc"] = d[kcol]
    d["st"] = np.where(d.av == 100, "COMPLETO", np.where(d.av == 0, "VACIO", "PARCIAL"))
    return d.drop_duplicates("doc", keep="last").set_index("doc")["st"]

docs_fic = set(fic[COL_DOC_FIC].dropna())
H = pstate(prog["Programa Hogares"], COL_DOC_FIC)
S = pstate(prog["Programa Salud"], COL_DOC_FIC)
E = pstate(prog["Programa Empresas"], COL_DOC_FIC)

# Tabla por ciudadano con el estado de los 4 ejes y su manzana (clave: IDMANZANA).
estado = U[["doc", "IDMANZANA", "mz_calidad"]].copy()
estado["ficha"] = np.where(estado.doc.isin(docs_fic), "COMPLETO", "VACIO")
estado["hogares"] = estado.doc.map(H).fillna("VACIO")
estado["salud"] = estado.doc.map(S).fillna("VACIO")
estado["empresas"] = estado.doc.map(E).fillna("VACIO")

# Puntaje de avance por eje por ciudadano: COMPLETO=1.0, PARCIAL=0.5, VACIO=0.0.
PESO = {"COMPLETO": 1.0, "PARCIAL": 0.5, "VACIO": 0.0}
for eje in EJES:
    estado[eje + "_sc"] = estado[eje].map(PESO)

# Solo ciudadanos con manzana asignada entran a la agregacion espacial.
con_mz = estado[estado["IDMANZANA"].notna()].copy()

def agregar_por_manzana(g):
    n = len(g)
    out = {"n_ciudadanos": n}
    out["n_aprox"] = int((g["mz_calidad"] == "APROXIMADA").sum())
    out["pct_aprox"] = round(100.0 * out["n_aprox"] / n, 1) if n else 0.0
    for eje in EJES:
        # % de avance del eje en la manzana = promedio del puntaje * 100.
        out["pct_" + eje] = round(float(g[eje + "_sc"].mean() * 100), 1) if n else 0.0
        out["n_" + eje + "_completo"] = int((g[eje] == "COMPLETO").sum())
        out["n_" + eje + "_parcial"] = int((g[eje] == "PARCIAL").sum())
    # Avance global de la manzana = promedio de los 4 ejes.
    out["pct_global"] = round(float(np.mean([out["pct_" + e] for e in EJES])), 1)
    return pd.Series(out)

agg = con_mz.groupby("IDMANZANA").apply(agregar_por_manzana).reset_index()

# GeoDataFrame de manzanas (en WGS84) con los agregados. Manzanas sin ciudadanos
# quedan con 0 y n_ciudadanos=0 (se grafican como "sin datos").
cols_pct = ["pct_ficha", "pct_hogares", "pct_salud", "pct_empresas", "pct_global"]
mz_av = mz_wgs.merge(agg, on="IDMANZANA", how="left")
for c in cols_pct:
    mz_av[c] = mz_av[c].fillna(0.0)
mz_av["n_ciudadanos"] = mz_av["n_ciudadanos"].fillna(0).astype(int)
mz_av["n_aprox"] = mz_av["n_aprox"].fillna(0).astype(int)
mz_av["pct_aprox"] = mz_av["pct_aprox"].fillna(0.0)
for eje in EJES:
    for suf in ("_completo", "_parcial"):
        col = "n_" + eje + suf
        mz_av[col] = mz_av[col].fillna(0).astype(int)

print("Ciudadanos con manzana:", len(con_mz), "en", con_mz["IDMANZANA"].nunique(), "manzanas")
print("Manzanas con al menos 1 ciudadano:", int((mz_av["n_ciudadanos"] > 0).sum()), "de", len(mz_av))
mz_av[["IDMANZANA", "n_ciudadanos"] + cols_pct].sort_values("n_ciudadanos", ascending=False).head(10)


### 11d. Exportacion de agregados (sin PII)

In [ ]:
# 11d. Exportar agregados por manzana (SIN PII) a data_processed_geo/.
# Solo se exportan conteos y porcentajes por manzana; nunca puntos de personas.
cols_exportar = ["IDMANZANA", "COMUNA", "BARRIO", "MANZANA", "n_ciudadanos",
                 "pct_ficha", "pct_hogares", "pct_salud", "pct_empresas", "pct_global"]
for eje in EJES:
    cols_exportar += ["n_" + eje + "_completo", "n_" + eje + "_parcial"]
mz_av_out = mz_av[cols_exportar + ["geometry"]].copy()

geojson_av = os.path.join(OUT_DIR, "avance_caracterizacion_manzana.geojson")
parquet_av = os.path.join(OUT_DIR, "avance_caracterizacion_manzana.parquet")
mz_av_out.to_file(geojson_av, driver="GeoJSON")
try:
    mz_av_out.to_parquet(parquet_av)
    print("Parquet:", parquet_av)
except Exception as e:
    print("Parquet no generado (requiere pyarrow):", e)
print("GeoJSON:", geojson_av)
print("Manzanas exportadas:", len(mz_av_out), "(solo agregados, sin PII)")


### 11e. Mapa interactivo del avance por eje

In [ ]:
# 11e. Mapa Folium del avance de caracterizacion por manzana.
# Mejoras aplicadas:
#   A) Dos salidas con el mismo codigo: modo "full" (interno) y "ejecutivo"
#      (compartible). El ejecutivo oculta el detalle de ubicacion.
#   B) MIN_N = 5: manzanas con < 5 ciudadanos muestran solo conteos, nunca
#      porcentajes (proteccion de datos ante muestras pequenas).
#   C) inset(): se encoge cada manzana ~5 m hacia su centroide para que se
#      vean separadas con borde blanco (lectura mas clara).
import folium

MIN_N = 5  # minimo de ciudadanos para publicar porcentajes

EJE_COLOR = {
    "ficha": OKI_VERDE,      # verde
    "hogares": OKI_NARANJA,  # naranja
    "salud": OKI_AZUL_C,     # azul cielo
    "empresas": OKI_AZUL,    # azul
}
EJE_LABEL = {"ficha": "Ficha", "hogares": "Hogares", "salud": "Salud", "empresas": "Empresas"}
GLOBAL_COLOR = OKI_MORADO

def _hex_a_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))

def gradiente(color_hex, pct):
    """Gradiente del color del eje: blanco (0%) -> color saturado (100%)."""
    r, g, b = _hex_a_rgb(color_hex)
    t = max(0.0, min(1.0, pct / 100.0))
    return "#%02X%02X%02X" % (int(255 + (r - 255) * t), int(255 + (g - 255) * t), int(255 + (b - 255) * t))

# C) inset: encoge la geometria ~5 m hacia su centroide (buffer negativo en metros).
# Se calcula en CRS metrico (EPSG:9377) y se devuelve en WGS84 para el mapa.
_INSET_M = 5.0
def inset(geom_wgs):
    try:
        g_m = gpd.GeoSeries([geom_wgs], crs=CRS_WEB).to_crs(CRS_METRIC).iloc[0]
        g_in = g_m.buffer(-_INSET_M)
        if g_in.is_empty:
            return geom_wgs
        return gpd.GeoSeries([g_in], crs=CRS_METRIC).to_crs(CRS_WEB).iloc[0]
    except Exception:
        return geom_wgs

# --- popup enriquecido por manzana (respeta MIN_N y el modo) ---
def svg_barras_ejes(pcts, colores, w=200, h=80):
    vmax = 100.0
    bw = w / (len(pcts) * 1.5)
    gap = bw * 0.5
    barras = []
    for i, (v, c) in enumerate(zip(pcts, colores)):
        bh = (v / vmax) * (h - 20)
        x = gap + i * (bw + gap)
        y = h - bh - 14
        barras.append(
            '<rect x="%.1f" y="%.1f" width="%.1f" height="%.1f" fill="%s" '
            'stroke="%s" stroke-width="0.5" rx="1"/>'
            '<text x="%.1f" y="%s" font-size="8" fill="%s" text-anchor="middle">%.0f</text>'
            % (x, y, bw, bh, c, OKI_GRIS, x + bw / 2, h - 2, OKI_GRIS, v)
        )
    return ('<svg width="%d" height="%d" style="background:%s;border-radius:4px">'
            % (w, h, BG)) + "".join(barras) + "</svg>"

def popup_av_html(r, modo="full"):
    n = int(r["n_ciudadanos"])
    publicar_pct = n >= MIN_N  # B) MIN_N
    cab = (
        '<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
        'background:%s;padding:8px;border-radius:6px;min-width:240px">'
        '<div style="font-size:13px;font-weight:700;color:%s">Manzana %s '
        '<span style="font-weight:400;color:#777">(%s)</span></div>'
        '<div style="color:#555">Comuna %s - Barrio %s</div>'
    ) % (BG, OKI_GRIS, r["MANZANA"], r["IDMANZANA"], r["COMUNA"], r["BARRIO"])

    if not publicar_pct:
        # Muestra solo conteos; nunca porcentajes.
        filas = "".join(
            '<tr><td style="padding:1px 6px">%s</td>'
            '<td style="text-align:right;padding:1px 6px"><b>%d</b> / %d</td></tr>'
            % (EJE_LABEL[e], int(r["n_" + e + "_completo"]), n)
            for e in EJES
        )
        cuerpo = (
            '<div style="margin:4px 0"><span style="background:#9AA3B2;color:#fff;'
            'padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700">'
            'Muestra pequena (%d &lt; %d)</span></div>'
            '<div style="font-size:10px;color:#666;margin-top:2px">Por proteccion de datos '
            'solo se muestran conteos (completos / total), no porcentajes.</div>'
            '<table style="font-size:11px;border-collapse:collapse;margin-top:4px;width:100%%">'
            '<tr style="color:%s;border-bottom:1px solid #ccc">'
            '<td style="padding:1px 6px">Eje</td>'
            '<td style="text-align:right;padding:1px 6px">completos / total</td></tr>%s</table>'
        ) % (n, MIN_N, OKI_GRIS, filas)
        return cab + cuerpo + "</div>"

    pcts = [r["pct_" + e] for e in EJES]
    cols = [EJE_COLOR[e] for e in EJES]
    svg = svg_barras_ejes(pcts, cols)
    filas = "".join(
        '<tr><td style="padding:1px 6px">%s</td>'
        '<td style="text-align:right;padding:1px 6px"><b>%.0f%%</b></td>'
        '<td style="text-align:right;padding:1px 6px;color:%s">%d / %d</td></tr>'
        % (EJE_LABEL[e], r["pct_" + e], OKI_GRIS,
           int(r["n_" + e + "_completo"]), n)
        for e in EJES
    )
    ubic = ""
    if modo == "full":
        ubic = (
            '<div style="font-size:10px;color:#666;margin-top:4px">Ubicacion: '
            '%d dentro del poligono, %d aprox. al borde (&lt;=25 m)</div>'
        ) % (n - int(r["n_aprox"]), int(r["n_aprox"]))
    cuerpo = (
        '<div style="margin:4px 0"><span style="background:%s;color:#fff;'
        'padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700">'
        'Avance global %.0f%%</span> '
        '<span style="color:#555;font-size:11px">%d ciudadanos ubicados</span></div>'
        '<div style="margin-top:6px;font-size:10px;color:%s;font-weight:700">'
        '%% de avance por eje</div>%s'
        '<table style="font-size:11px;border-collapse:collapse;margin-top:4px;width:100%%">'
        '<tr style="color:%s;border-bottom:1px solid #ccc">'
        '<td style="padding:1px 6px">Eje</td>'
        '<td style="text-align:right;padding:1px 6px">%% avance</td>'
        '<td style="text-align:right;padding:1px 6px">completos</td></tr>%s</table>%s'
    ) % (GLOBAL_COLOR, r["pct_global"], n, OKI_GRIS, svg, OKI_GRIS, filas, ubic)
    return cab + cuerpo + "</div>"

# Precalcular las geometrias con inset una sola vez (optimizacion).
mz_av = mz_av.copy()
mz_av["geom_inset"] = mz_av["geometry"].apply(inset)

def construir_mapa_avance(modo):
    """modo: 'full' (interno) o 'ejecutivo' (compartible)."""
    centro_av = [mz_wgs.geometry.union_all().centroid.y, mz_wgs.geometry.union_all().centroid.x]
    m_av = folium.Map(location=centro_av, zoom_start=16, tiles=None, control_scale=True,
                   zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
        attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m_av)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_av)
    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
        attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_av)

    contorno_av = folium.FeatureGroup(name="Contorno territorial", show=True)
    folium.GeoJson(
        mz_wgs.to_json(),
        style_function=lambda f: {"color": OKI_GRIS, "weight": 1.2, "fill": False},
    ).add_to(contorno_av)
    contorno_av.add_to(m_av)
    # Capa de SOMBRA: predios en gris translucido (contexto catastral), sin popup.
    if TERRENOS_ZONA is not None:
        sombra_fg = folium.FeatureGroup(name="Predios (sombra)", show=False)
        folium.GeoJson(
            TERRENOS_ZONA.to_json(),
            style_function=lambda f: {"fillColor": "#333333", "color": "#555555",
                                      "weight": 0.3, "fillOpacity": 0.18},
        ).add_to(sombra_fg)
        sombra_fg.add_to(m_av)

    def add_coropleta(nombre, campo_pct, color_hex):
        fg = folium.FeatureGroup(name=nombre, show=False)
        for _, r in mz_av.iterrows():
            n = int(r["n_ciudadanos"])
            pct = r[campo_pct]
            # B) sin pct publicable -> gris neutro aunque haya ciudadanos
            if n == 0:
                fill = "#E8E8E8"
            elif n < MIN_N:
                fill = "#D6D9DF"
            else:
                fill = gradiente(color_hex, pct)
            dash = "4 3" if (n >= MIN_N and r["pct_aprox"] > 50) else None
            # C) usar geometria con inset para separacion visual; borde blanco
            folium.GeoJson(
                r["geom_inset"].__geo_interface__,
                style_function=(lambda f, c=fill, d=dash: {"fillColor": c, "color": "#FFFFFF",
                                                    "weight": 1.2, "fillOpacity": 0.85,
                                                    "dashArray": d}),
                popup=folium.Popup(popup_av_html(r, modo), max_width=300),
                tooltip=("Manzana %s" % r["MANZANA"]) if n < MIN_N else
                        ("Manzana %s - %s %.0f%%" % (r["MANZANA"], campo_pct.replace("pct_", ""), pct)),
            ).add_to(fg)
        fg.add_to(m_av)

    for eje in EJES:
        add_coropleta("Avance " + EJE_LABEL[eje], "pct_" + eje, EJE_COLOR[eje])
    add_coropleta("Avance global", "pct_global", GLOBAL_COLOR)

    leyenda_ejes = "".join(
        '<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
        '<i style="width:14px;height:14px;background:%s;border:1px solid %s;'
        'display:inline-block;border-radius:3px"></i><span>%s</span></div>'
        % (EJE_COLOR[e], OKI_GRIS, EJE_LABEL[e]) for e in EJES
    )
    leyenda_ejes += (
        '<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
        '<i style="width:14px;height:14px;background:%s;border:1px solid %s;'
        'display:inline-block;border-radius:3px"></i><span>Global</span></div>'
        % (GLOBAL_COLOR, OKI_GRIS)
    )
    etiqueta_modo = "Interno (full)" if modo == "full" else "Ejecutivo (compartible)"
    legend_av = (
        '<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:%s;'
        'padding:10px 12px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
        'font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:%s">'
        '<div style="font-weight:700;margin-bottom:4px">Avance de caracterizacion</div>'
        '<div style="font-size:10px;color:#888;margin-bottom:4px">Version: %s</div>%s'
        '<div style="font-size:10px;color:#666;margin-top:4px">Intensidad = %% de avance '
        '(blanco 0%% a color 100%%). Gris claro = muestra &lt; %d (solo conteos). '
        'Gris = sin ciudadanos. Borde punteado = ubicacion aproximada.</div></div>'
    ) % (BG, OKI_GRIS, etiqueta_modo, leyenda_ejes, MIN_N)
    m_av.get_root().html.add_child(folium.Element(legend_av))

    if POLIGONO_ZONA is not None:
        zona_fg = folium.FeatureGroup(name="Poligono Ciudad Paraiso", show=True)
        folium.GeoJson(
            POLIGONO_ZONA.to_json(),
            style_function=lambda f: {"color": "#8A6D1A", "weight": 5,
                                      "dashArray": "10 6", "fill": False},
            tooltip="Zona de intervencion: Ciudad Paraiso",
        ).add_to(zona_fg)
        zona_fg.add_to(m_av)

    if POLIGONO_ZONA is not None:
        m_av.fit_bounds([[3.442445, -76.53259], [3.44791, -76.526004]], padding=(30, 30))
    folium.LayerControl(collapsed=False, position="topright").add_to(m_av)
    return m_av

# A) generar las dos salidas con el mismo codigo
m_full = construir_mapa_avance("full")
agregar_export(m_full, "Avance caracterizacion", "avance")
html_full = os.path.join(OUT_DIR, "mapa_avance_caracterizacion.html")
m_full.save(html_full)

m_ejec = construir_mapa_avance("ejecutivo")
agregar_export(m_ejec, "Avance caracterizacion (ejecutivo)", "avance_ejecutivo")
html_ejec = os.path.join(OUT_DIR, "mapa_avance_caracterizacion_ejecutivo.html")
m_ejec.save(html_ejec)

print("Mapas exportados:")
print("  full (interno):   ", html_full)
print("  ejecutivo (comp.):", html_ejec)
print("MIN_N =", MIN_N, "| manzanas con < MIN_N ciudadanos (solo conteos):",
      int(((mz_av["n_ciudadanos"] > 0) & (mz_av["n_ciudadanos"] < MIN_N)).sum()))
m_full


### 11f. Resumen analitico

In [ ]:
# 11f. Resumen analitico del avance de caracterizacion.
total_ciudadanos = int(len(U))
total_ubicados = int(U["lat"].notna().sum())
total_con_mz = int(U["IDMANZANA"].notna().sum())
pct_ubicados = round(100 * total_ubicados / total_ciudadanos, 1) if total_ciudadanos else 0.0
pct_con_mz = round(100 * total_con_mz / total_ciudadanos, 1) if total_ciudadanos else 0.0

# Avance promedio por eje sobre los ciudadanos con manzana asignada.
avance_prom_eje = {EJE_LABEL[e]: round(float(con_mz[e + "_sc"].mean() * 100), 1) if len(con_mz) else 0.0
                   for e in EJES}

resumen_av = {
    "ciudadanos_totales": total_ciudadanos,
    "ciudadanos_ubicados": total_ubicados,
    "pct_ubicados": pct_ubicados,
    "ciudadanos_con_manzana": total_con_mz,
    "pct_con_manzana": pct_con_mz,
    "manzanas_con_ciudadanos": int((mz_av["n_ciudadanos"] > 0).sum()),
    "avance_promedio_por_eje": avance_prom_eje,
}
print(json.dumps(resumen_av, ensure_ascii=False, indent=2))

MIN_CIUDADANOS = 3  # minimo para que el ranking no lo dominen manzanas con 1 registro
print("\nManzanas con MENOR avance global (con al menos %d ciudadanos):" % MIN_CIUDADANOS)
con_datos = mz_av[mz_av["n_ciudadanos"] >= MIN_CIUDADANOS].sort_values("pct_global").head(3)
if con_datos.empty:
    con_datos = mz_av[mz_av["n_ciudadanos"] > 0].sort_values("pct_global").head(3)
for _, r in con_datos.iterrows():
    print("  - Manzana %s (%s), Comuna %s Barrio %s: avance global %.0f%% | "
          "Ficha %.0f%% Hogares %.0f%% Salud %.0f%% Empresas %.0f%% | %d ciudadanos"
          % (r["MANZANA"], r["IDMANZANA"], r["COMUNA"], r["BARRIO"], r["pct_global"],
             r["pct_ficha"], r["pct_hogares"], r["pct_salud"], r["pct_empresas"],
             int(r["n_ciudadanos"])))

print("\nNota: el avance por manzana solo considera ciudadanos geolocalizados y")
print("asignados a una manzana catastral. Ciudadanos sin GPS ni direccion parseable")
print("no entran en la agregacion espacial. Datos reales de la encuesta (sin PII en el mapa).")


## 12. Seguridad: mapa de calor y serie de tiempo

Datos reales de la Dimension Seguridad (comparendos, hurtos, homicidios)
unificados en `data/geojson/seguridad_unificado.geojson`. El heatmap usa un
gradiente perceptualmente uniforme (INFERNO) accesible para daltonismo. Los
homicidios (muestra pequena y sensible) se representan de forma agregada y, si
se muestran como punto, sin ningun dato personal.

In [ ]:
# 12a. Carga del GeoJSON unificado de seguridad y la serie mensual.
#       Resolucion de rutas robusta (local y Google Colab).
def _resolver(nombre, subcarpeta):
    cand = [
        os.path.join("..", "data", subcarpeta, nombre),
        os.path.join("data", subcarpeta, nombre),
        os.path.join("/content", "data", subcarpeta, nombre),
        os.path.join("/content", "drive", "MyDrive",
                     "mapa_Analitico_ciudad_Paraiso", "data", subcarpeta, nombre),
    ]
    for p in cand:
        if os.path.isfile(p):
            return os.path.abspath(p)
    return None

_seg_path = _resolver("seguridad_unificado.geojson", "geojson")
if _seg_path is None:
    raise FileNotFoundError(
        "No se encontro seguridad_unificado.geojson. Ejecuta primero el script "
        "de unificacion para generar data/geojson/seguridad_unificado.geojson.")

seg = gpd.read_file(_seg_path).to_crs(CRS_WEB)
seg["fecha"] = pd.to_datetime(seg["fecha"], errors="coerce")
# lat/lon desde geometria por si el archivo no los trajera consistentes.
seg["lon"] = seg.geometry.x
seg["lat"] = seg.geometry.y

# Serie mensual: vive en data_processed_geo/ (no en data/). Cargar o reconstruir.
_serie_path = None
for p in [os.path.join("..", "data_processed_geo", "seguridad_serie_mensual.csv"),
          os.path.join("data_processed_geo", "seguridad_serie_mensual.csv"),
          os.path.join("/content", "data_processed_geo", "seguridad_serie_mensual.csv")]:
    if os.path.isfile(p):
        _serie_path = os.path.abspath(p)
        break

if _serie_path and os.path.isfile(_serie_path):
    serie_mensual = pd.read_csv(_serie_path)
else:
    serie_mensual = (seg.dropna(subset=["fecha"])
                        .assign(periodo=lambda d: d["fecha"].dt.strftime("%Y-%m"))
                        .groupby(["periodo", "categoria"]).size()
                        .reset_index(name="conteo"))

CATS_SEG = ["hurto", "comparendo", "homicidio"]
print("Eventos de seguridad:", len(seg))
print(seg["categoria"].value_counts().to_string())
print("Serie mensual filas:", len(serie_mensual),
      "| periodos:", serie_mensual["periodo"].nunique())

In [ ]:
# 12b. Mapa Folium de seguridad: heatmap por categoria (Blues/Greens/Reds) + capas de puntos.
import folium
from folium.plugins import HeatMap

# Gradiente de calor POR CATEGORIA (rampas secuenciales ColorBrewer, aptas
# para daltonismo: cada capa varia en luminosidad dentro de un mismo matiz,
# intenso = oscuro). Asi cada capa se distingue por su color.
GRAD_SEG = {
    "hurto":      {0.0: "#eff3ff", 0.4: "#9ecae1", 0.7: "#4292c6", 1.0: "#08306b"},  # Blues
    "comparendo": {0.0: "#edf8e9", 0.4: "#a1d99b", 0.7: "#41ab5d", 1.0: "#00441b"},  # Greens
    "homicidio":  {0.0: "#fee5d9", 0.4: "#fcae91", 0.7: "#fb6a4a", 1.0: "#67000d"},  # Reds
}
GRAD_DEFAULT = {0.0: "#f0f0f0", 0.5: "#969696", 1.0: "#252525"}  # grises (fallback)
# Color de punto por categoria (Okabe-Ito, accesible).
SEG_PUNTO = {"hurto": OKI_AZUL, "comparendo": OKI_VERDE, "homicidio": OKI_BERMELL}

# Centro del mapa: centroide de la zona si existe, si no de los puntos.
if POLIGONO_ZONA is not None:
    _c = POLIGONO_ZONA.geometry.union_all().centroid
    centro_seg = [_c.y, _c.x]
else:
    centro_seg = [seg["lat"].mean(), seg["lon"].mean()]

m_seg = folium.Map(location=centro_seg, zoom_start=16, tiles=None, control_scale=True,
                   zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
# Mapa a pantalla completa: el contenedor ocupa el 100%% de la ventana.
m_seg.get_root().width = "100%"
m_seg.get_root().height = "100%"
_css_full = ("<style>html,body{height:100%;margin:0;padding:0}"
             ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;"
             "width:100%;height:100%}</style>")
m_seg.get_root().header.add_child(folium.Element(_css_full))

# 3 capas base conmutables (Google Streets activa). No usar OpenStreetMap.
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m_seg)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_seg)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_seg)

# Mascara oscura FUERA de la zona (poligono mundial con agujero = la zona) para enfocar.
if POLIGONO_ZONA is not None:
    _zona_geom = POLIGONO_ZONA.geometry.union_all()
    _mundo = [[-180.0, -85.0], [180.0, -85.0], [180.0, 85.0], [-180.0, 85.0], [-180.0, -85.0]]
    _agujeros = []
    _geoms = list(_zona_geom.geoms) if _zona_geom.geom_type == "MultiPolygon" else [_zona_geom]
    for _g in _geoms:
        _agujeros.append([[x, y] for x, y in _g.exterior.coords])
    _mascara = {"type": "Feature", "properties": {},
                "geometry": {"type": "Polygon", "coordinates": [_mundo] + _agujeros}}
    folium.GeoJson(
        _mascara,
        name="Mascara (foco en la zona)",
        style_function=lambda f: {"fillColor": "#000000", "color": "#000000",
                                  "weight": 0, "fillOpacity": 0.25},
        control=False,
    ).add_to(m_seg)

# Puntos solo dentro de la zona (para heatmap y capas de puntos).
if POLIGONO_ZONA is not None:
    _zona = POLIGONO_ZONA.geometry.union_all()
    seg_z = seg[seg.geometry.within(_zona) | seg.geometry.intersects(_zona)].copy()
else:
    seg_z = seg.copy()
# Capas de PUNTOS: solo eventos con fecha >= 2025-09-01 (no afecta calor ni graficas).
_FECHA_CORTE_PUNTOS = pd.Timestamp("2025-09-01")
seg_z_pts = seg_z[pd.to_datetime(seg_z["fecha"], errors="coerce") >= _FECHA_CORTE_PUNTOS].copy()

# Un HeatMap por categoria, conmutable. Se construyen y se guardan por categoria;
# el add_to(m_seg) se hace al final en el orden deseado del control de capas.
_fg_calor = {}
for cat in CATS_SEG:
    pts = seg_z[seg_z["categoria"] == cat][["lat", "lon"]].dropna().values.tolist()
    if not pts:
        continue
    fg = folium.FeatureGroup(name="Calor: " + cat + " (" + str(len(pts)) + ")",
                             show=False)
    HeatMap(pts, gradient=GRAD_SEG.get(cat, GRAD_DEFAULT), radius=18, blur=14, min_opacity=0.35,
            name="heat_" + cat).add_to(fg)
    _fg_calor[cat] = fg

# Capa de puntos hurtos/comparendos (CircleMarker con popup SIN PII). Apagada al inicio.
def _popup_evento(r):
    f = r["fecha"].strftime("%Y-%m-%d") if pd.notna(r["fecha"]) else "sin fecha"
    sub = (r["subtipo"] or "").strip()
    sit = (r.get("sitio") or "").strip()
    filas = '<div style="font-weight:700;color:%s">%s</div>' % (OKI_GRIS, r["categoria"].capitalize())
    if sub:
        filas += '<div>Tipo: %s</div>' % sub
    filas += '<div>Fecha: %s</div>' % f
    if sit:
        filas += '<div>Sitio: %s</div>' % sit
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:160px">%s</div>'
            % (BG, filas))

_fg_puntos = {}
for cat in ["hurto", "comparendo"]:
    sub = seg_z_pts[seg_z_pts["categoria"] == cat]
    fg = folium.FeatureGroup(name="Puntos: " + cat + " (" + str(len(sub)) + ")", show=False)
    for _, r in sub.iterrows():
        if pd.isna(r["lat"]) or pd.isna(r["lon"]):
            continue
        folium.CircleMarker(
            location=[r["lat"], r["lon"]], radius=3,
            color=SEG_PUNTO[cat], weight=1, fill=True,
            fill_color=SEG_PUNTO[cat], fill_opacity=0.7,
            popup=folium.Popup(_popup_evento(r), max_width=240),
        ).add_to(fg)
    _fg_puntos[cat] = fg

# Homicidios como puntos con popup MINIMO (solo fecha), sin datos personales. Apagada.
_hom = seg_z_pts[seg_z_pts["categoria"] == "homicidio"]
fg_hom = folium.FeatureGroup(name="Puntos: homicidio (" + str(len(_hom)) + ")", show=False)
for _, r in _hom.iterrows():
    if pd.isna(r["lat"]) or pd.isna(r["lon"]):
        continue
    f = r["fecha"].strftime("%Y-%m-%d") if pd.notna(r["fecha"]) else "sin fecha"
    pop = ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
           'background:%s;padding:6px 8px;border-radius:6px">'
           '<div style="font-weight:700;color:%s">Homicidio</div>'
           '<div>Fecha: %s</div></div>' % (BG, OKI_BERMELL, f))
    folium.CircleMarker(
        location=[r["lat"], r["lon"]], radius=4,
        color=OKI_BERMELL, weight=1, fill=True,
        fill_color=OKI_BERMELL, fill_opacity=0.75,
        popup=folium.Popup(pop, max_width=180),
    ).add_to(fg_hom)
# fg_hom se agrega al final, en el orden deseado del control de capas.

# Poligono de la zona (borde dorado punteado), encima de todo.


# --- Capas del dataset ITT consolidado (data/1_data_itt), conmutables y apagadas ---
def _ruta_itt(nombre):
    cands = [
        __import__("os").path.join("..", "data", "1_data_itt", nombre),
        __import__("os").path.join("data", "1_data_itt", nombre),
        __import__("os").path.join("/content", "data", "1_data_itt", nombre),
        __import__("os").path.join("/content", "drive", "MyDrive",
                   "mapa_Analitico_ciudad_Paraiso", "data", "1_data_itt", nombre),
    ]
    for _p in cands:
        if __import__("os").path.isfile(_p):
            return __import__("os").path.abspath(_p)
    return None

def _add_geojson_itt(nombre, capa_nombre, style, tooltip_field=None, popup_fn=None, to_points=False, show=False):
    _rp = _ruta_itt(nombre)
    if _rp is None:
        print("Aviso: no se encontro", nombre, "- capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    # Convertir columnas de fecha/timestamp a texto (folium no serializa Timestamp).
    for _col in _g.columns:
        if _col == "geometry":
            continue
        if str(_g[_col].dtype).startswith("datetime"):
            _g[_col] = _g[_col].astype(str)
    _fg = folium.FeatureGroup(name=capa_nombre, show=show)
    if to_points:
        for _, _r in _g.iterrows():
            if _r.geometry is None:
                continue
            _pt = _r.geometry
            folium.CircleMarker(
                location=[float(_pt.y), float(_pt.x)], radius=4,
                color="#333333", weight=1, fill=True, fill_color="#D55E00", fill_opacity=0.75,
                popup=(folium.Popup(popup_fn(_r), max_width=240) if popup_fn else None),
            ).add_to(_fg)
    else:
        folium.GeoJson(
            _g.to_json(),
            style_function=(lambda f, s=style: s),
            tooltip=(folium.GeoJsonTooltip(fields=[tooltip_field]) if tooltip_field and tooltip_field in _g.columns else None),
        ).add_to(_fg)
    print("Capa ITT agregada:", capa_nombre, "(", len(_g), "features )")
    return _fg

def _popup_hom300(r):
    f = r.get("fecha")
    try:
        f = f.strftime("%Y-%m-%d")
    except Exception:
        f = str(f)
    d = r.get("dist_a_estacion_m")
    arma = str(r.get("tipo_arma") or "s/d")
    movil = str(r.get("movil") or "s/d")
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:170px">'
            '<div style="font-weight:700;color:#D55E00">Homicidio (&lt;=300 m estacion)</div>'
            '<div>Fecha: %s</div><div>Distancia: %s m</div>'
            '<div>Arma: %s</div><div>Movil: %s</div></div>'
            % (BG, f, (int(d) if d is not None else "s/d"), arma, movil))

# 01 manzanas con ITT consolidado: tooltip/popup con 4 atributos clave y area formateada.
def _add_manzanas_itt():
    _rp = _ruta_itt("01_manzanas_itt_consolidado.geojson")
    if _rp is None:
        print("Aviso: no se encontro manzanas consolidado - capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    # Area metrica real en m2 (CRS metrico EPSG:9377). Si existe shape_area se ignora
    # para homogeneidad; se recalcula desde la geometria.
    _g["area_m2"] = _g.to_crs(CRS_METRIC).geometry.area
    # Formato legible: entero con separador de miles en punto (ej. 14.520 m2).
    _g["area_m2_fmt"] = _g["area_m2"].apply(lambda x: ("{:,}".format(int(round(x))).replace(",", ".")) + " m2")
    # Campos estandarizados.
    _g["id_manzana"] = _g["idmanzana"].astype(str)
    _g["barrio_f"] = _g["barrio"].astype(str)
    _g["manzana_f"] = _g["manzana"].astype(str)
    _fg = folium.FeatureGroup(name="ITT: manzanas (consolidado)", show=False)
    _tt = folium.GeoJsonTooltip(
        fields=["id_manzana", "barrio_f", "manzana_f", "area_m2_fmt"],
        aliases=["ID Manzana:", "Barrio:", "Manzana:", "Area:"],
        localize=True, sticky=True,
        style=("background-color:%s;color:%s;font-family:system-ui,Segoe UI,sans-serif;"
               "font-size:12px;padding:6px 8px;border-radius:6px;border:1px solid #ccc;" % (BG, OKI_GRIS)),
    )
    def _popup_mz(idm, bar, man, af):
        return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
                'background:%s;padding:6px 8px;border-radius:6px;min-width:170px">'
                '<div style="font-weight:700;color:%s;margin-bottom:3px">Manzana %s</div>'
                '<table style="font-size:11px;border-collapse:collapse">'
                '<tr><td style="color:#666;padding:1px 6px">ID Manzana</td><td style="padding:1px 6px"><b>%s</b></td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Barrio</td><td style="padding:1px 6px">%s</td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Manzana</td><td style="padding:1px 6px">%s</td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Area</td><td style="padding:1px 6px"><b>%s</b></td></tr>'
                '</table></div>' % (BG, OKI_GRIS, man, idm, bar, man, af))
    folium.GeoJson(
        _g.to_json(),
        style_function=lambda f: {"fill": False, "color": "#333333",
                                  "weight": 1.0},
        tooltip=_tt,
    ).add_to(_fg)
    # Popups por manzana (tabla limpia).
    for _, _r in _g.iterrows():
        folium.GeoJson(
            _r["geometry"].__geo_interface__,
            style_function=lambda f: {"fillColor": "#56B4E9", "color": "#333333",
                                      "weight": 0.0, "fillOpacity": 0.0},
            popup=folium.Popup(_popup_mz(_r["id_manzana"], _r["barrio_f"],
                                         _r["manzana_f"], _r["area_m2_fmt"]), max_width=240),
        ).add_to(_fg)
    print("Capa ITT agregada: manzanas (consolidado) (", len(_g), "manzanas )")
    return _fg
# Demoliciones Cristo Rey: clasificar por estado de demolicion (columna 'demolicion').
DEMOL_COLOR = {
    "Demolido": "#D55E00",                       # bermellon: ya demolido
    "Programado para demolicion": "#E69F00",     # naranja: programado
    "En proceso (hacia demolicion)": "#F0E442",  # amarillo: en proceso
    "Sin demolicion programada": "#009E73",      # verde: conservado / sin intervencion
}
def _color_demol(v):
    return DEMOL_COLOR.get(str(v).strip(), "#999999")
def _popup_demol(r):
    est = str(r.get("demolicion") or "s/d")
    est2 = str(r.get("estado") or "")
    barrio = str(r.get("barrio") or "")
    npn = str(r.get("npn") or r.get("predio_catastral") or "")
    fdem = r.get("fecha_demolicion")
    try:
        fdem = fdem.strftime("%Y-%m-%d")
    except Exception:
        fdem = "" if fdem is None else str(fdem)
    filas = '<div style="font-weight:700;color:%s">Predio - %s</div>' % (_color_demol(est), est)
    if est2 and est2 != est:
        filas += '<div>Estado catastral: %s</div>' % est2
    if barrio:
        filas += '<div>Barrio: %s</div>' % barrio
    if npn:
        filas += '<div>Predio: %s</div>' % npn
    if fdem:
        filas += '<div>Fecha demolicion: %s</div>' % fdem
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:180px">%s</div>'
            % (BG, filas))
def _add_demoliciones():
    _rp = _ruta_itt("10_demoliciones_cristo_rey_predios.geojson")
    if _rp is None:
        print("Aviso: no se encontro demoliciones - capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    for _c in _g.columns:
        if _c != "geometry" and str(_g[_c].dtype).startswith("datetime"):
            _g[_c] = _g[_c].astype(str)
    _fg = folium.FeatureGroup(name="ITT: demoliciones Cristo Rey", show=False)
    for _, _r in _g.iterrows():
        _col = _color_demol(_r.get("demolicion"))
        folium.GeoJson(
            _r["geometry"].__geo_interface__,
            style_function=(lambda f, c=_col: {"fillColor": c, "color": "#333333",
                                               "weight": 0.8, "fillOpacity": 0.65}),
            tooltip="Demolicion: %s" % str(_r.get("demolicion") or "s/d"),
            popup=folium.Popup(_popup_demol(_r), max_width=260),
        ).add_to(_fg)
    print("Capa ITT agregada: demoliciones Cristo Rey (", len(_g), "predios )")
    return _fg

# --- Construccion de las capas ITT (sin agregarlas aun al mapa) ---
# 00 limite de la zona (UNICO limite oficial; estilo homologado del poligono). Activo al inicio.
_fg_limite = _add_geojson_itt("00_limite_ciudad_paraiso.geojson", "Limite Ciudad Paraiso",
                              {"color": "#8A6D1A", "weight": 5, "dashArray": "10 6", "fill": False},
                              show=True)
# 01 manzanas con ITT consolidado.
_fg_manzanas = _add_manzanas_itt()
# 10 demoliciones Cristo Rey (poligonos de fondo).
_fg_demol = _add_demoliciones()
# 11 estacion de policia Fray Damian (manzana): solo contorno, sin relleno.
_fg_estacion = _add_geojson_itt("11_estacion_fray_damian_manzana.geojson", "ITT: estacion Fray Damian",
                                {"color": "#0072B2", "weight": 2.75, "fill": False},
                                tooltip_field="nombre")
# 12 homicidios <=300m de la estacion (puntos, popup SIN PII)
_fg_hom300 = _add_geojson_itt("12_estacion_fray_damian_homicidios_300m.geojson",
                              "ITT: homicidios <=300m estacion", {}, popup_fn=_popup_hom300, to_points=True)

# --- Orden final de overlays en el control de capas (de primera a ultima) ---
# El LayerControl lista los overlays en el mismo orden en que se hace add_to(m_seg).
# 1. Limite Ciudad Paraiso (activo)  2. ITT manzanas  3. ITT estacion Fray Damian
# 4. Puntos y calor + homicidios <=300m  6. demoliciones (poligono de fondo) al final.
_orden_seg = [
    _fg_limite,
    _fg_manzanas,
    _fg_estacion,
    _fg_puntos.get("hurto"),
    _fg_puntos.get("comparendo"),
    fg_hom,
    _fg_hom300,
    _fg_calor.get("hurto"),
    _fg_calor.get("comparendo"),
    _fg_calor.get("homicidio"),
    _fg_demol,
]
for _capa in _orden_seg:
    if _capa is not None:
        _capa.add_to(m_seg)

if POLIGONO_ZONA is not None:
    m_seg.fit_bounds([[3.442445, -76.53259], [3.44791, -76.526004]], padding=(30, 30))
folium.LayerControl(collapsed=False, position="topright").add_to(m_seg)
print("Mapa de seguridad construido. Puntos en zona:", len(seg_z))

In [ ]:
# 12c. Panel flotante colapsable: resumen + serie de tiempo (Chart.js con fallback a tabla).
#       Se inyecta en el HTML del mapa con get_root().html.add_child(...).

# Datos de la serie para el grafico: periodos ordenados y una serie por categoria.
# Serie ANUAL: se agrega por anio (primeros 4 chars del periodo YYYY-MM).
_serie_anual = serie_mensual.copy()
_serie_anual["anio"] = _serie_anual["periodo"].astype(str).str[:4]
_periodos = sorted(_serie_anual["anio"].dropna().unique().tolist())
_pivot = (_serie_anual.pivot_table(index="anio", columns="categoria",
                                   values="conteo", aggfunc="sum")
          .reindex(_periodos).fillna(0).astype(int))
for _c in CATS_SEG:
    if _c not in _pivot.columns:
        _pivot[_c] = 0
_series_cat = {c: _pivot[c].tolist() for c in CATS_SEG}

# Colores de linea accesibles (Okabe-Ito).
_LINEA = {"hurto": OKI_AZUL, "comparendo": OKI_VERDE, "homicidio": OKI_BERMELL}

# Totales por categoria para el resumen.
_tot_cat = seg["categoria"].value_counts().to_dict()
_resumen_items = "".join(
    '<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    '<i style="width:12px;height:12px;background:%s;display:inline-block;border-radius:2px"></i>'
    '<span>%s: <b>%d</b></span></div>'
    % (_LINEA[c], c.capitalize(), int(_tot_cat.get(c, 0))) for c in CATS_SEG
)

# Tabla HTML de respaldo (si Chart.js no carga).
_filas_tabla = ""
for p in _periodos:
    celdas = "".join('<td style="padding:1px 5px;text-align:right">%d</td>' % int(_pivot.loc[p, c]) for c in CATS_SEG)
    _filas_tabla += '<tr><td style="padding:1px 5px">%s</td>%s</tr>' % (p, celdas)
_cab_tabla = "".join('<th style="padding:1px 5px">%s</th>' % c.capitalize() for c in CATS_SEG)
_tabla_html = (
    '<table style="font-size:10px;border-collapse:collapse;width:100%%">'
    '<tr style="border-bottom:1px solid #ccc"><th style="padding:1px 5px">Ano</th>%s</tr>%s</table>'
    % (_cab_tabla, _filas_tabla))

# Datos como JSON embebido (consumidos por el script del panel).
_chart_data = json.dumps({
    "labels": _periodos,
    "series": [{"label": c.capitalize(), "data": _series_cat[c], "color": _LINEA[c],
             "axis": ("y2" if c == "comparendo" else "y")} for c in CATS_SEG],
}, ensure_ascii=False)

_panel = """
<div id="seg-panel" style="position:fixed;top:20px;left:20px;z-index:9999;
     background:__BG__;max-width:480px;width:min(480px,46vw);
     padding:10px 12px;border-radius:10px;box-shadow:0 2px 10px rgba(0,0,0,.3);
     font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="font-weight:700;margin-bottom:4px">Seguridad - resumen</div>
  __RESUMEN__
  <details style="margin-top:6px">
    <summary style="cursor:pointer;font-weight:600;color:__AZUL__;outline:none">
      Ver serie de tiempo (anual) / Ocultar</summary>
    <div style="margin-top:8px">
      <div style="font-size:9px;color:#666;margin-bottom:4px">Doble escala: eje izquierdo = hurtos y homicidios; eje derecho = comparendos (magnitud mayor).</div>
      <div style="position:relative;height:200px;width:100%"><canvas id="seg-chart"></canvas></div>
      <div id="seg-fallback" style="display:none;max-height:160px;overflow:auto">__TABLA__</div>
    </div>
  </details>
</div>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
(function(){
  var DATA = __DATA__;
  function dibujar(){
    var cv = document.getElementById('seg-chart');
    var fb = document.getElementById('seg-fallback');
    if (typeof Chart === 'undefined' || !cv){
      if (cv) cv.style.display='none';
      if (fb) fb.style.display='block';
      return;
    }
    try {
      new Chart(cv.getContext('2d'), {
        type: 'line',
        data: {
          labels: DATA.labels,
          datasets: DATA.series.map(function(s){
            return {label: s.label, data: s.data, borderColor: s.color,
                    backgroundColor: s.color, borderWidth: 2, pointRadius: 2,
                    tension: 0.25, fill: false, yAxisID: (s.axis||'y')};
          })
        },
        options: {responsive:true, maintainAspectRatio:false, interaction:{mode:'index',intersect:false},
          plugins:{legend:{labels:{boxWidth:10,font:{size:10}}}},
          scales:{x:{ticks:{font:{size:9}}},
                  y:{type:'linear',position:'left',beginAtZero:true,
                     title:{display:true,text:'Cantidad de casos (hurtos / homicidios)',font:{size:9}},
                     ticks:{font:{size:9}}},
                  y2:{type:'linear',position:'right',beginAtZero:true,
                      title:{display:true,text:'Cantidad de comparendos',font:{size:9}},
                      grid:{drawOnChartArea:false},ticks:{font:{size:9}}}}}
      });
    } catch(e){
      if (cv) cv.style.display='none';
      if (fb) fb.style.display='block';
    }
  }
  if (document.readyState === 'complete'){ setTimeout(dibujar, 300); }
  else { window.addEventListener('load', function(){ setTimeout(dibujar, 300); }); }
})();
</script>
"""
_panel = (_panel.replace("__BG__", BG).replace("__GRIS__", OKI_GRIS)
                .replace("__AZUL__", OKI_AZUL).replace("__RESUMEN__", _resumen_items)
                .replace("__TABLA__", _tabla_html).replace("__DATA__", _chart_data))
m_seg.get_root().html.add_child(folium.Element(_panel))
print("Panel colapsable inyectado (resumen + serie de tiempo).")
# --- Control deslizante de intensidad del heatmap (runtime, no cambia la base) ---
_MAP_ID = m_seg.get_name()
_slider_html = """
<div id="heat-intensity-ctrl" style="position:fixed;bottom:24px;right:24px;z-index:9999;
     background:__BG__;padding:8px 12px;border-radius:10px;
     box-shadow:0 2px 10px rgba(0,0,0,.3);font-family:system-ui,Segoe UI,sans-serif;
     font-size:12px;color:__GRIS__;max-width:220px">
  <div style="font-weight:700;margin-bottom:4px">Intensidad del calor</div>
  <input id="heat-intensity-range" type="range" min="8" max="40" value="18"
         style="width:100%;cursor:pointer">
  <div style="font-size:10px;color:#666;margin-top:2px">
    Radio: <span id="heat-intensity-val">18</span> px</div>
</div>
<script>
(function(){
  function aplicar(valor){
    var mp = window["__MAP_ID__"];
    if (!mp || typeof mp.eachLayer !== "function"){ return; }
    var r = parseInt(valor, 10);
    var b = Math.round(r * 0.8);
    mp.eachLayer(function(l){
      if (l && typeof l.setOptions === "function" && l._latlngs){
        try {
          l.setOptions({radius: r, blur: b});
          if (typeof l.redraw === "function"){ l.redraw(); }
        } catch(e){}
      }
    });
    var vs = document.getElementById("heat-intensity-val");
    if (vs){ vs.textContent = r; }
  }
  function init(){
    var inp = document.getElementById("heat-intensity-range");
    if (!inp){ return; }
    inp.addEventListener("input", function(){ aplicar(this.value); });
  }
  if (document.readyState === "complete"){ setTimeout(init, 400); }
  else { window.addEventListener("load", function(){ setTimeout(init, 400); }); }
})();
</script>
"""
_slider_html = (_slider_html.replace("__BG__", BG).replace("__GRIS__", OKI_GRIS)
                            .replace("__MAP_ID__", _MAP_ID))
m_seg.get_root().html.add_child(folium.Element(_slider_html))
print("Slider de intensidad del heatmap inyectado.")


In [ ]:
# 12d. Exportar el mapa de seguridad a data_processed_geo/.
agregar_export(m_seg, "Seguridad", "seguridad")
html_seg = os.path.join(OUT_DIR, "mapa_seguridad_calor.html")
m_seg.save(html_seg)
print("Mapa de seguridad exportado:", html_seg)
m_seg

In [ ]:
# 12e. Resumen de seguridad: totales por categoria, meses cubiertos y top 3 meses.
_tot = seg["categoria"].value_counts()
print("Total de eventos por categoria:")
for c in CATS_SEG:
    print("  %-11s %d" % (c, int(_tot.get(c, 0))))
print("Total general:", int(len(seg)))

_fval = seg["fecha"].dropna()
if len(_fval):
    print("Rango de fechas:", _fval.min().date(), "->", _fval.max().date())
print("Meses cubiertos:", serie_mensual["periodo"].nunique())

_top = (seg.dropna(subset=["fecha"])
           .assign(periodo=lambda d: d["fecha"].dt.strftime("%Y-%m"))
           .groupby("periodo").size().sort_values(ascending=False).head(3))
print("Top 3 meses con mas incidentes:")
for p, n in _top.items():
    print("  %s -> %d" % (p, int(n)))